# 키워드 확정 루프 — 증거 기반 (흥행 *유발* 검증)

> 목적: 파라미터 조정 → 나온 키워드가 **진짜 흥행을 유발**하는지 4축 증거로 확인 → 확정 → 대시보드 연결.
> 4축: ① 통계(purity·성공률) ② **인과(Δprob: 더하면 성공확률 오르나)** ③ 실증(실매출) ④ 지지도.

```
[튜닝] EngineConfig 임계 조정 → 장부 재생성
   ↓
[검토] evidence_table 로 Δprob·매출 스캔 → 단일 키워드 drill-down
   ↓  (반복)
[확정] keyword_final.csv 저장 (include/tag + 증거)
   ↓
[연결] python -m scripts.export_dashboard → dashboard.html 반영
```
**당신이 하는 일**: 셀 돌리며 Δprob≤0인 가짜 killer 골라내고, 임계 조정해 재확인 → 마지막에 CSV 저장.

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.max_rows", 200)
from src.eval.md.engine import MDEngine, EngineConfig
from src.eval.md import inspector as I
from src.eval.md.inspector import keyword_evidence, evidence_table, ledger_keywords, export_keyword_final

MODEL = "v2_sweepA"   # "v2_sweepA"(최종 채택) | "exp47"  (장부는 모델 거의 불변)
print("model:", MODEL)

## 1. 엔진 로드 + 장부 (파라미터는 EngineConfig)

In [2]:
# ★ 파라미터 튜닝 지점 — 값 바꾸고 이 셀부터 다시 실행하면 장부가 바뀜
CFG = (EngineConfig.exp47() if MODEL=="exp47" else EngineConfig.v2_sweepA())
CFG.killer_purity = 0.50   # killer purity 하한 (base 0.238 위)
CFG.killer_top_q  = 0.75   # Score_succ 상위 25%
CFG.mine_purity   = 0.15   # mine purity 상한
CFG.hub_top_q     = 0.80   # Hub_Score 상위 20%
CFG.hub_balance_eps = 0.15 # |purity-base| 균형 게이트

eng = MDEngine(CFG).run_single_inference(); eng.build_mass(); lg = eng.build_ledger("full")
print(f"killer {len(lg.killer)} / mine {len(lg.mine)} / hub {len(lg.hub)} / neutral {eng.cache['K']-len(lg.killer|lg.mine|lg.hub)}")

killer 62 / mine 131 / hub 144 / neutral 1726


## 2. 증거 스캔 — killer가 진짜 흥행을 유발하나 (Δprob)

In [3]:
killers = ledger_keywords(eng, "killer")
tbl = evidence_table(eng, killers)
# Δprob>0 = 흥행 유발(인과 확인) / Δprob≤0 = 상관만 (가짜 killer 의심)
tbl = tbl.sort_values("delta_prob_mean", ascending=False)
print(f"killer {len(tbl)}개 중 Δprob>0(진짜 유발): {(tbl['delta_prob_mean']>0).sum()}개 / Δprob≤0(상관만): {(tbl['delta_prob_mean']<=0).sum()}개")
display(tbl)

killer 62개 중 Δprob>0(진짜 유발): 31개 / Δprob≤0(상관만): 31개


,keyword,tag,성공률,base_rate,purity,balance,support_succ,support_fail,delta_prob_mean,delta_prob_pos_rate,매출중앙값,hub_score
54,하와이,killer,0.833,0.238,0.833,0.595,10,2,0.2542,1.00,NaN,-0.865
32,알포트,killer,0.833,0.238,0.833,0.595,5,1,0.2220,1.00,4394468.0,1.039
25,뷰티,killer,1.000,0.238,0.875,0.637,7,2,0.1997,1.00,NaN,-0.410
34,여유,killer,1.000,0.238,0.800,0.562,7,2,0.1684,1.00,NaN,-0.419
58,효모,killer,1.000,0.238,0.578,0.340,6,27,0.1665,1.00,2297363.0,-1.282
27,빈츠,killer,0.800,0.238,0.800,0.562,4,1,0.1501,1.00,6592740.0,0.770
50,포카칩,killer,1.000,0.238,1.000,0.762,3,0,0.1369,1.00,2811004.0,0.878
45,커스터드,killer,0.611,0.238,0.511,0.273,69,133,0.1187,1.00,2378001.0,2.959
6,고창,killer,0.800,0.238,0.800,0.562,4,1,0.1179,1.00,6686735.0,1.546
46,콜라겐,killer,1.000,0.238,0.750,0.512,6,3,0.1155,1.00,1402147.0,-1.574


In [5]:
# mine / hub 도 같은 방식
print("── MINE (Δprob<0 이어야 진짜 악재) ──"); display(evidence_table(eng, ledger_keywords(eng,"mine")).sort_values("delta_prob_mean"))
print("── HUB (balance 작아야 진짜 일반어) ──"); display(evidence_table(eng, ledger_keywords(eng,"hub")).sort_values("balance"))

── MINE (Δprob<0 이어야 진짜 악재) ──


,keyword,tag,성공률,base_rate,purity,balance,support_succ,support_fail,delta_prob_mean,delta_prob_pos_rate,매출중앙값,hub_score
58,순수,mine,0.000,0.238,0.079,0.159,3,18,-0.1844,0.00,NaN,3.632
116,피규어,mine,0.000,0.238,0.142,0.096,19,50,-0.1742,0.00,5000.0,1.150
64,식후,mine,0.000,0.238,0.095,0.143,4,19,-0.1661,0.07,NaN,0.437
33,밀키트,mine,0.000,0.238,0.149,0.088,13,24,-0.1613,0.00,15010.0,-0.715
122,해장,mine,0.000,0.238,0.141,0.097,49,135,-0.1612,0.00,11700.0,-0.914
118,한우,mine,0.000,0.238,0.000,0.238,0,14,-0.1592,0.00,21850.0,-1.663
97,축구,mine,0.000,0.238,0.000,0.238,0,5,-0.1517,0.00,NaN,-2.247
35,벚꽃,mine,0.000,0.238,0.000,0.238,0,4,-0.1502,0.00,1283162.0,-1.250
111,펭귄,mine,0.000,0.238,0.000,0.238,0,13,-0.1464,0.00,1516671.0,-0.534
124,향신료,mine,0.000,0.238,0.125,0.113,6,22,-0.1430,0.00,263915.0,-0.748


── HUB (balance 작아야 진짜 일반어) ──


,keyword,tag,성공률,base_rate,purity,balance,support_succ,support_fail,delta_prob_mean,delta_prob_pos_rate,매출중앙값,hub_score
119,토스트,hub,0.235,0.238,0.235,0.003,4,13,-0.0293,0.53,1759448.0,1.206
118,키치,hub,NaN,0.238,0.241,0.003,13,41,-0.0614,0.47,NaN,1.637
40,맛,hub,0.333,0.238,0.244,0.006,29,88,-0.0108,0.60,7660100.0,1.846
134,피자,hub,0.204,0.238,0.245,0.007,33,103,-0.0192,0.73,5889520.0,1.925
96,젤리,hub,0.314,0.238,0.246,0.008,126,270,-0.0467,0.33,630070.0,3.678
79,스파게티,hub,0.154,0.238,0.246,0.008,8,18,-0.0775,0.00,6839252.0,2.954
35,리본,hub,0.000,0.238,0.229,0.009,11,16,-0.0522,0.20,NaN,2.961
3,간식,hub,0.338,0.238,0.229,0.009,968,2332,0.0215,1.00,2680139.0,1.860
14,김밥,hub,0.318,0.238,0.229,0.009,98,207,-0.0743,0.07,15445631.0,1.172
26,도시락,hub,0.241,0.238,0.249,0.011,406,1048,-0.1767,0.00,1040000.0,2.708


## 3. 단일 키워드 drill-down (실제 제품·매출 확인)

In [4]:
SEED = "고창"   # 의심스러운 키워드 넣어 확인
e = keyword_evidence(eng, SEED)
for k,v in e.items():
    if k != "예시제품": print(f"  {k}: {v}")
display(e["예시제품"])

  keyword: 고창
  n_carriers: 5
  성공률: 0.8
  base_rate: 0.238
  tag: killer
  purity: 0.8
  balance: 0.562
  support_succ: 4
  support_fail: 1
  hub_score: 1.546
  delta_prob_mean: 0.0736
  delta_prob_pos_rate: 1.0
  n: 25
  매출중앙값: 6686735.0


,product,성공,prob,매출30d
0,롯데)빈츠고창꿀고구마102g,성공,0.926,6390282.0
1,롯데)카스타드고창꿀고구마210g,성공,0.901,4522851.0
2,롯데)찰떡파이고창꿀고구마250g,성공,0.757,7554925.0
3,롯데)마가렛트고창꿀고구마176g,성공,0.596,6983188.0
4,롯데)말랑카우고창꿀고구마158g,실패,0.335,NaN


## 3.5 캐리어별 절제 분해 — 상호작용(modifier) 판정

`keyword_evidence`의 Δprob는 캐리어 평균이라 상호작용을 뭉갠다. 여기선 **실제 보유 제품마다** 키워드를 빼본 기여(`contrib`)를 분리 → "비스킷엔 +, 캔디엔 ≈0" 같은 조건부 효과가 드러난다.
- `contrib` ≫ 0: 이 제품에서 진짜 일함 / ≈0: 무의미 / <0: 악재
- `keyword_disentangle`: 동반 키워드(예: 고창↔꿀고구마) 중 실제 드라이버 분리

In [ ]:
# 3.5 캐리어별 in-context 절제 — 같은 키워드가 '무엇에 붙느냐'로 성공이 갈리는지
from src.eval.md.inspector import keyword_context_breakdown, keyword_disentangle

bd = keyword_context_breakdown(eng, SEED)
print(f"[{SEED}] 보유 {len(bd)}개 제품 · contrib = 이 제품에서 키워드를 빼면 떨어지는 성공확률 (클수록 진짜 일함)")
display(bd)

# 교란 분리 — 동반 키워드 중 진짜 신호의 주인은? (예: 고창 vs 꿀고구마)
print(f"\n[{SEED}] 교란 분리 (동반키워드별 평균 기여 비교):")
display(keyword_disentangle(eng, SEED))

## 4. 확정 export → keyword_final.csv

In [ ]:
# 전 키워드 + 태그 + Δprob 증거 + 추천액션. (태그 키워드 Δprob 계산에 수 분)
df = export_keyword_final(eng)
print("저장: data/processed/hin/keyword_final.csv |", len(df), "키워드")
print("추천액션:"); print(df[df.tag!="neutral"]["suggested"].value_counts().to_string())
print("\n→ 이 CSV에서 include(Y/N)·tag 손보세요. '강등검토'는 Δprob≤0인 가짜 killer 후보입니다.")

## 5. 대시보드 연결 (확정 후)
CSV 확정(include/tag 편집) 후 터미널에서:
```
python -m scripts.export_dashboard
```
→ `Dashboard/config.js` 재생성 → `dashboard.html` 열면 확정 키워드·태그(색) 반영. **코드 수정 없음.**